In [1]:
import os
import time
import subprocess
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import pandas as pd

In [2]:
def compressImg(input_path, output_path):
    """
    Compress an img using FLIF.
    """
    cmd = ['flif', '-e', '--overwrite', input_path, output_path]
    try:
        subprocess.run(cmd, check=True)
    except Exception as e:
        print(f'Error compressing {input_path}: {e}.')
        return False
    
    return True


def decompressImg(input_path, output_path):
    """
    DeCompress an FLIF img.
    """
    cmd = ['flif', '-d', '--overwrite', input_path, output_path]
    try:
        subprocess.run(cmd, check=True)
    except Exception as e:
        print(f'Error decompressing {input_path}: {e}.')
        return False
    
    return True

In [3]:
def cal_compression_ratio(original_path, compressed_path):
    """
    Calculate compression rate based on file sizes.
    """
    original_size = os.path.getsize(original_path)
    compressed_size = os.path.getsize(compressed_path)

    with Image.open(original_path) as img:
        width, height = img.size
        npixels = width * height
    bpsp = (compressed_size * 8) / (npixels * 3)
    compression_ratio = compressed_size / original_size
    
    return compression_ratio, original_size, compressed_size, bpsp, width, height, npixels

In [4]:
def process_dataset(dataset_path, output_dir, mode="compress", cal_metric=True, verbose=False, keywords=None):
    """
    Process a dataset (Kodak or CLIC) using FLIF, with optional compression rate calculation.
    """
    dataset_path = Path(dataset_path)
    output_dir   = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    imgs = list(dataset_path.glob('*.png'))
    if keywords is not None:
        imgs = [img for img in imgs if keywords in img.name]
        
    if len(imgs) == 0:
        print(f'No Imgs found in {dataset_path}.')
        return
    
    rows = []
    for img in imgs:
        output_path = output_dir / f'{img.stem}.flif' if mode == 'compress' else output_dir / f'{img.stem}_recon.png'
        
        start = time.time()
        if mode == 'compress':
            success = compressImg(img, output_path)
        elif mode == 'decompress':
            success = decompressImg(img, output_path)
        elapsed = time.time() - start   
          
        if success and cal_metric and mode == 'compress':
            compression_ratio, original_size, compressed_size, bpsp, width, height, npixels = cal_compression_ratio(img, output_path)
            
            if verbose:
                print(f'{img.name}: compression ratio = {compression_ratio:.2%} (Original: {original_size} bytes, Compressed: {compressed_size} bytes), bpsp: {bpsp}.')
            rows.append([img.name, bpsp, compression_ratio, original_size, compressed_size, width, height, npixels, elapsed])
        
    return rows

In [5]:
""" 1. compress Touch and Go (png) """
dataset_path = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/compressed/flif'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_TouchandGoDataset-v2.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_TouchandGoDataset-v2.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,1400.000000,1400.000000,1400.000000,1400.000000,1400.0,1400.0,1400.0,1400.000000,1400.000000
mean,0.808398,0.319125,288034.526429,93127.406429,640.0,480.0,307200.0,0.678193,2.500300
std,0.220826,0.037750,44205.507025,25439.135763,0.0,0.0,0.0,0.041369,0.383728
min,0.397352,0.237980,192348.000000,45775.000000,640.0,480.0,307200.0,0.597174,1.669687
25%,0.676845,0.294367,253517.500000,77972.500000,640.0,480.0,307200.0,0.651774,2.200673
50%,0.748190,0.307641,276483.000000,86191.500000,640.0,480.0,307200.0,0.667341,2.400026
75%,0.829032,0.332857,310647.000000,95504.500000,640.0,480.0,307200.0,0.687416,2.696589
max,1.931997,0.473848,481097.000000,222566.000000,640.0,480.0,307200.0,0.877957,4.176189


In [6]:
""" 2. compress Objectfolder (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/compressed/flif'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_ObjectFolder_1.0.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_ObjectFolder_1.0.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.0,20000.0,20000.0,20000.000000,20000.000000
mean,3.764793,0.948106,28541.906000,27106.509850,160.0,120.0,19200.0,0.059622,3.964154
std,0.429817,0.023452,2695.165442,3094.685281,0.0,0.0,0.0,0.006028,0.374329
min,2.957083,0.874015,24187.000000,21291.000000,160.0,120.0,19200.0,0.048612,3.359306
25%,3.435556,0.930909,26490.000000,24736.000000,160.0,120.0,19200.0,0.054280,3.679167
50%,3.628750,0.944407,27698.000000,26127.000000,160.0,120.0,19200.0,0.060065,3.846944
75%,4.058194,0.965346,30118.000000,29219.000000,160.0,120.0,19200.0,0.063454,4.183056
max,5.607500,1.050346,41962.000000,40374.000000,160.0,120.0,19200.0,0.207747,5.828056


In [7]:
""" 3. compress SSVTP (png) """
dataset_path = '/data/ssd/zhaoy/datasets/SSVTP/dataset-comp/test'
output_dir = '/data/ssd/zhaoy/datasets/SSVTP/compressed/flif'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_SSVTP.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_SSVTP.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,918.000000,918.000000,918.000000,918.000000,918.0,918.0,918.0,918.000000,918.000000
mean,1.566461,0.700475,64301.678649,45114.064270,240.0,320.0,76800.0,0.181139,2.232697
std,0.152949,0.016650,4853.413152,4404.924101,0.0,0.0,0.0,0.010577,0.168521
min,1.270660,0.663180,53899.000000,36595.000000,240.0,320.0,76800.0,0.166004,1.871493
25%,1.430825,0.687335,60208.000000,41207.750000,240.0,320.0,76800.0,0.174981,2.090556
50%,1.547726,0.700187,63670.500000,44574.500000,240.0,320.0,76800.0,0.179512,2.210781
75%,1.680990,0.714191,67915.000000,48412.500000,240.0,320.0,76800.0,0.184773,2.358160
max,1.953437,0.738121,77399.000000,56259.000000,240.0,320.0,76800.0,0.275470,2.687465


In [8]:
""" 4. compress ObjTac (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjTac/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/ObjTac/compressed/flif'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_ObjTac.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_ObjTac.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,51.000000,51.000000,51.000000,51.000000,51.0,51.000000,51.000000,51.000000,51.000000
mean,0.363413,0.555025,11784.803922,7175.235294,60.0,903.352941,54201.176471,0.269718,0.578609
std,0.247838,0.168329,7995.848535,5083.346845,0.0,391.917891,23515.073434,0.163623,0.328989
min,0.009479,0.149144,818.000000,122.000000,60.0,331.000000,19860.000000,0.032037,0.063559
25%,0.132395,0.470490,4868.500000,1951.500000,60.0,587.500000,35250.000000,0.153139,0.253969
50%,0.376885,0.601010,10936.000000,7490.000000,60.0,813.000000,48780.000000,0.216022,0.629227
75%,0.540626,0.639515,19075.500000,11768.500000,60.0,1142.000000,68520.000000,0.381542,0.826285
max,0.908166,1.093870,28278.000000,16135.000000,60.0,1802.000000,108120.000000,0.698667,1.246181


In [11]:
""" 4. compress YCB-Slide (png) """
dataset_path = '/data/ssd/zhaoy/datasets/YCB-Slide/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/YCB-Slide/compressed/flif'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_YCB-Slide.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/flif_YCB-Slide.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,10916.000000,10916.000000,10916.000000,10916.000000,10916.0,10916.0,10916.0,10916.000000,10916.000000
mean,1.489446,0.681852,62881.925247,42896.043881,240.0,320.0,76800.0,0.177962,2.183400
std,0.082957,0.009080,2789.584906,2389.174080,0.0,0.0,0.0,0.005078,0.096861
min,1.271181,0.653149,54243.000000,36610.000000,240.0,320.0,76800.0,0.166277,1.883438
25%,1.431875,0.675530,60928.750000,41238.000000,240.0,320.0,76800.0,0.174253,2.115582
50%,1.482448,0.681080,62656.000000,42694.500000,240.0,320.0,76800.0,0.177044,2.175556
75%,1.535243,0.687339,64545.250000,44215.000000,240.0,320.0,76800.0,0.181298,2.241155
max,1.828576,0.716257,73586.000000,52663.000000,240.0,320.0,76800.0,0.219151,2.555069
